# 27 — Reservoir Sampling (Amazon FAR style)

Sample `k` items uniformly from a stream of **unknown** length in one pass, O(k) memory. Parts 1-3
build the core (concurrency at Part 3); Parts 4-5 are stretch tasks (merging partition samples, then
parallel sampling). RNG is **seeded** so tests are deterministic. Fill stubs, run each test cell, peek
at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Algorithm R

`reservoir_sample(stream, k, seed) -> list`: keep the first `k` items; for the `i`-th item (`i >= k`)
pick `j = randint(0, i)` and, if `j < k`, replace `reservoir[j]`. Use `random.Random(seed)`.

**Lock down:** every item ends up with probability `k/n`; one pass, no need to know `n`; with fewer
than `k` items you return them all.

In [ ]:
import random


def reservoir_sample(stream, k, seed):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    out = reservoir_sample(range(1000), 10, seed=42)
    assert len(out) == 10 and len(set(out)) == 10 and all(0 <= x < 1000 for x in out)
    assert sorted(reservoir_sample([1, 2, 3], 5, seed=1)) == [1, 2, 3]    # fewer than k
    assert reservoir_sample(range(1000), 10, seed=42) == out             # deterministic
    print("Part 1 OK")

_t1()

## Part 2 — Streaming reservoir

`Reservoir(k, seed)` with `add(item)` and `sample()` — the incremental form of Part 1 (you don't have
the whole stream at once). Feeding a stream item-by-item must match `reservoir_sample` with the same
seed.

**Lock down:** keep a running count `n`; the per-item logic is identical to Algorithm R; `sample()`
returns a copy.

In [ ]:
class Reservoir:
    def __init__(self, k, seed):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def sample(self):
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    r = Reservoir(10, seed=42)
    for x in range(1000):
        r.add(x)
    assert r.sample() == reservoir_sample(range(1000), 10, seed=42)       # same as the batch version
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe reservoir

`ThreadSafeReservoir(k, seed)`: `add`/`sample` safe under many producer threads feeding the stream.

**The invariant:** 8 threads each add 100 distinct items; the final sample has exactly `k` distinct
items, all drawn from what was added. **Discuss:** the count + replace is a read-modify-write that
needs the lock; uniformity still holds across threads (each `add` is one logical stream step).

In [ ]:
import threading


class ThreadSafeReservoir:
    def __init__(self, k, seed):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def sample(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    r = ThreadSafeReservoir(10, seed=0)

    def worker(b):
        for i in range(100):
            r.add("t%d_%d" % (b, i))

    ts = [threading.Thread(target=worker, args=(b,)) for b in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    s = r.sample()
    added = {"t%d_%d" % (b, i) for b in range(8) for i in range(100)}
    assert len(s) == 10 and len(set(s)) == 10 and all(x in added for x in s)
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Merge partition samples

`merge(res_a, count_a, res_b, count_b, k, seed) -> list`: combine two reservoirs (each a uniform sample
of `count_a` / `count_b` items) into one size-`k` sample of the union — pick each output slot from A
with probability `count_a / (count_a + count_b)`, else from B.

**Discuss:** this is what lets you sample partitions independently and combine; weighting by the
partition sizes keeps the merged sample uniform over the union.

In [ ]:
def merge(res_a, count_a, res_b, count_b, k, seed):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    a, b = [1, 2, 3, 4, 5], [10, 20, 30, 40, 50]
    out = merge(a, 500, b, 500, 5, seed=1)
    assert len(out) == 5 and all(x in (set(a) | set(b)) for x in out)
    assert merge(a, 500, b, 500, 5, seed=1) == out                       # deterministic
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel sampling

`parallel_sample(stream, k, num_procs, seed) -> list`: split the stream across processes, reservoir-
sample each chunk in parallel (worker `reservoir_workers.sample_chunk`), then `merge` the partition
samples down to `k`.

**Discuss:** sampling is per-chunk independent (parallel), then a weighted merge combines them; the
seeds are offset per chunk for independence; this is how you sample a dataset too big for one machine.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import reservoir_workers


def parallel_sample(stream, k, num_procs, seed):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    stream = list(range(1000))
    out = parallel_sample(stream, 10, num_procs=4, seed=7)
    assert len(out) == 10 and all(x in set(stream) for x in out)
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Weighted reservoir sampling (A-Res: key = u^(1/w), keep top-k); sampling with replacement.
- Sliding-window / time-decayed sampling; distinct-sample vs with-duplicates.
- Proving uniformity of Algorithm R by induction; reservoir of size k vs k independent samplers.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import random
import threading
from concurrent.futures import ProcessPoolExecutor
import reservoir_workers


def ref_reservoir_sample(stream, k, seed):
    rng = random.Random(seed)
    res = []
    for i, x in enumerate(stream):
        if i < k:
            res.append(x)
        else:
            j = rng.randint(0, i)
            if j < k:
                res[j] = x
    return res


class RefReservoir:
    def __init__(self, k, seed):
        self.k = k
        self.rng = random.Random(seed)
        self.res = []
        self.n = 0

    def add(self, item):
        i = self.n
        self.n += 1
        if i < self.k:
            self.res.append(item)
        else:
            j = self.rng.randint(0, i)
            if j < self.k:
                self.res[j] = item

    def sample(self):
        return list(self.res)


class RefThreadSafeReservoir:
    def __init__(self, k, seed):
        self.k = k
        self.rng = random.Random(seed)
        self.res = []
        self.n = 0
        self.lock = threading.Lock()

    def add(self, item):
        with self.lock:
            i = self.n
            self.n += 1
            if i < self.k:
                self.res.append(item)
            else:
                j = self.rng.randint(0, i)
                if j < self.k:
                    self.res[j] = item

    def sample(self):
        with self.lock:
            return list(self.res)


def ref_merge(res_a, count_a, res_b, count_b, k, seed):
    rng = random.Random(seed)
    total = count_a + count_b
    out = []
    for _ in range(k):
        if res_a and rng.random() * total < count_a:
            out.append(res_a[rng.randrange(len(res_a))])
        elif res_b:
            out.append(res_b[rng.randrange(len(res_b))])
        elif res_a:
            out.append(res_a[rng.randrange(len(res_a))])
    return out


def ref_parallel_sample(stream, k, num_procs, seed):
    stream = list(stream)
    chunks = [stream[i::num_procs] for i in range(num_procs)]
    items = [(c, k, seed + i) for i, c in enumerate(chunks)]
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        partials = list(ex.map(reservoir_workers.sample_chunk, items))
    res, count = partials[0]
    mseed = seed
    for r, c in partials[1:]:
        res = ref_merge(res, count, r, c, k, mseed)
        count += c
        mseed += 1
    return res


_o = ref_reservoir_sample(range(100), 5, 3)
assert len(_o) == 5 and ref_reservoir_sample(range(100), 5, 3) == _o
_r = RefReservoir(5, 3)
for _x in range(100):
    _r.add(_x)
assert _r.sample() == _o
_m = ref_merge([1, 2], 10, [3, 4], 10, 3, 0); assert len(_m) == 3 and all(x in {1, 2, 3, 4} for x in _m)
assert len(ref_parallel_sample(range(500), 7, 4, 1)) == 7
print("reference solutions OK")